This notebook aims to generate relevant graphs for visualisation concerns 

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np 

import torch

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset 


import torchvision 
import torchvision.transforms as transforms 

from sklearn.cluster import KMeans 
from sklearn.neighbors import NearestNeighbors

from typing import List, Tuple 


In [6]:
def score_least_confidence(logits):
    probs = torch.softmax(logits ,  dim=1)
    return ( 1- probs.max(dim=1).values)

In [7]:
def score_margin(logits):
    probs = torch.softmax(logits,dim=1)
    top2 = torch.topk(probs, 2, dim=1).values

    return top2[:,0] - top2[:,1] 


In [8]:
def score_entropy(logits):
    probs = torch.softmax(logits,dim=1)
    return - 1 * ( probs * probs.log()).sum(dim=1)

In [9]:
def compute_badge_embeddings(model, loader, device):
    model.eval()
    grads = [] 



    for x, _ in loader:
        x = x.to(device)
        x.requires_grad=True

        logits =model(x)
        probs = torch.softmax(logits, dim=1)

        y_hat = probs.argmax(dim=1)



        loss = F.cross_entropy(logits, y_hat, reduction='sum')
        loss.backward()


        grads.append(x.grad.view(x.size(0), -1).cpu().numpy())
    return np.concatenate(grads, axis=0)

In [10]:
def select_by_uncertainty(model, unlabeled_loader, device, B, scoring_fn):
    model.eval()
    scores = []

    with torch.no_grad():
        for x, idx in unlabeled_loader:
            x=x.to(device)
            logits = model(x)
            s =  scoring_fn(logits)
            scores.extend(list(zip(idx.numpy(), s.cpu(). numpy()) ))

    # desc. uncertainity
    scores = sorted(scores, key=lambda x: -x[1])
    selected = [idx for idx, _ in scores[:B]]

    return selected 

In [11]:
def plot_baseline_comparison(results_dict, budgets):
    plt.figure(figsize=(8,5))
    for method, accs in results_dict.items():
        plt.plot(budgets, accs, marker="x", label=method)

    plt.xlabel("Budget")
    plt.ylabel("Accuracy")

    plt.title("Active Learning baseline comparison")

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

Active Learning Loop Section 

In [ ]:
def active_learning_round(
    model,
    train_dataset,
    test_loader,
    labeled_indices,
    unlabeled_indices,
    B,
    scoring_fn,
    device
):
    # 1. Train classifier on current labelled set
    labeled_subset = torch.utils.data.Subset(train_dataset, labeled_indices)
    loader = DataLoader(labeled_subset, batch_size=128, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(20):   # 20 epochs is enough
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # 2. Score unlabelled pool
    unlabeled_subset = torch.utils.data.Subset(train_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

    selected = select_by_uncertainty(model, unlabeled_loader, device, B, scoring_fn)

    # Convert subset indices back to global indices
    selected_global = [unlabeled_indices[i] for i in selected]

    # 3. Update labelled/unlabelled sets
    new_labeled = labeled_indices + selected_global
    new_unlabeled = [i for i in unlabeled_indices if i not in selected_global]

    # 4. Evaluate accuracy
    acc = evaluate(model, test_loader, device)

    return new_labeled, new_unlabeled, acc
